In [107]:
# import necessary dependancies
import pandas as pd
from pymongo import MongoClient
from pprint import pprint

In [108]:
host="localhost"
port = 27017

In [109]:
# connect to the database
client=MongoClient(f"mongodb://{host}:{port}")

In [110]:
db =client["passengers"]

In [111]:
db.list_collection_names()

['air_passengers']

In [112]:
collection=db["air_passengers"]

In [113]:
collection.find_one()

{'_id': ObjectId('6ae18ed8150052860f19470e'),
 'month': datetime.datetime(1949, 1, 1, 0, 0),
 'total_passengers': 112}

- how does the data look like
- data type
- null values
- duplicates 
- number of rows and columns

In [114]:
# get first five observation
observations=collection.find().limit(5)

for observation in observations:
    pprint(observation)

{'_id': ObjectId('6ae18ed8150052860f19470e'),
 'month': datetime.datetime(1949, 1, 1, 0, 0),
 'total_passengers': 112}
{'_id': ObjectId('6ae18ed8150052860f19470f'),
 'month': datetime.datetime(1949, 2, 1, 0, 0),
 'total_passengers': 118}
{'_id': ObjectId('6ae18ed8150052860f194710'),
 'month': datetime.datetime(1949, 3, 1, 0, 0),
 'total_passengers': 132}
{'_id': ObjectId('6ae18ed8150052860f194711'),
 'month': datetime.datetime(1949, 4, 1, 0, 0),
 'total_passengers': 129}
{'_id': ObjectId('6ae18ed8150052860f194712'),
 'month': datetime.datetime(1949, 5, 1, 0, 0),
 'total_passengers': 121}


In [115]:
# count number of documents
collection.count_documents({})

144

There are 144 observations

In [116]:
#get the one document
document=collection.find_one()

# check for data type
for field,value in document.items():
    print(field,type(value))

_id <class 'bson.objectid.ObjectId'>
month <class 'datetime.datetime'>
total_passengers <class 'int'>


In [117]:
# check for null values
documents =collection.find()

for doc in documents:
    for field,value in doc.items():
        if value is None:
            print(f"Null found in field: {field}, document: {doc['_id']}")
            

In [118]:
df=pd.DataFrame(list(collection.find()))

In [119]:
df.isna().sum()

_id                 0
month               0
total_passengers    0
dtype: int64

No missing values

In [120]:
# check for duplicates

pipeline=[
    {
        "$group":{
            "_id":{
                "month":"$month",
                "passengers":"$passengers"
            },
            "count":{"$sum":1}
        }
      
    },
      {
                "$match":{"count":{"$gt":1}}
            }
]

In [121]:
duplicates =list(collection.aggregate(pipeline))
duplicates

[]

No duplicates

In [122]:
df =pd.DataFrame(list(collection.find({},{"_id":0})))
df

,month,total_passengers
0,1949-01-01,112
1,1949-02-01,118
2,1949-03-01,132
3,1949-04-01,129
4,1949-05-01,121
...,...,...
139,1960-08-01,606
140,1960-09-01,508
141,1960-10-01,461
142,1960-11-01,390


In [123]:
%%writefile mongo_wrangle.py
import pandas as pd
from pymongo import MongoClient
def get_data(port,host,db,col):
    client=MongoClient(f"mongodb://{host}:{port}")
    database=client[db]
    collection=database[col]
    df =pd.DataFrame(list(collection.find({},{"_id":0})))
    client.close()
    return df


    

Overwriting mongo_wrangle.py
